In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pmdarima import auto_arima

import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
client_id = 

df = (
    spark.table(
        "corealgos.eup_level_useful_account_balances_euro"
    )
    .filter(
        F.col("eup_grid_id") == client_id
    )
    .groupby("date")
    .agg(
        F.sum("balance_in_euro").alias(
            "balance_in_euro"
        )
    )
    .toPandas()
)

print(df.shape)
df.head()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
df["date"] = pd.to_datetime(df["date"])

df = (
    df.sort_values("date")
      .set_index("date")
      .asfreq("D")
)

df["balance_in_euro"] = (
    df["balance_in_euro"]
      .ffill()
      .fillna(0)
)

series = (
    df["balance_in_euro"]
      .astype(float)
      .values
)

print("Length:", len(series))
print("Mean:", np.mean(series))
print("Std:", np.std(series))

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(
    df.index,
    series
)

plt.title(
    f"Client {client_id} Balance"
)

plt.show()

In [ ]:
def make_sliding_windows(
    y,
    history=100,
    horizon=28
):

    X = []
    Y = []

    for t in range(
        history,
        len(y) - horizon + 1
    ):

        X.append(
            y[t-history:t]
        )

        Y.append(
            y[t:t+horizon]
        )

    return (
        np.array(X),
        np.array(Y)
    )

In [ ]:
def naive_forecast(
    history,
    horizon
):

    return np.repeat(
        history[-1],
        horizon
    )

In [ ]:
def fit_autoarima(
    history,
    horizon=28,
    actual_future=None,
    seasonal=True,
    m=7,
    alpha=0.1
):

    # ==========================================
    # Fit AutoARIMA
    # ==========================================

    model = auto_arima(
        history,
        seasonal=seasonal,
        m=m,
        start_p=0,
        start_q=0,
        max_p=5,
        max_q=5,
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        trace=False,
        n_fits=50
    )

    # ==========================================
    # In-sample predictions
    # ==========================================

    fitted, train_conf = model.predict_in_sample(
        return_conf_int=True,
        alpha=alpha
    )

    train_rmse = np.sqrt(
        np.mean(
            (history - fitted) ** 2
        )
    )

    # ==========================================
    # Train Winkler Score
    # ==========================================

    train_winkler_scores = []

    for y, l, u in zip(
        history,
        train_conf[:, 0],
        train_conf[:, 1]
    ):

        width = u - l

        if y < l:

            score = (
                width +
                (2 / alpha) * (l - y)
            )

        elif y > u:

            score = (
                width +
                (2 / alpha) * (y - u)
            )

        else:

            score = width

        train_winkler_scores.append(score)

    train_winkler = np.mean(
        train_winkler_scores
    )

    # ==========================================
    # Forecast
    # ==========================================

    forecast, conf = model.predict(
        n_periods=horizon,
        return_conf_int=True,
        alpha=alpha
    )

    # ==========================================
    # Result Dictionary
    # ==========================================

    result = {

        "history": history,

        "forecast": forecast,

        "lower": conf[:, 0],
        "upper": conf[:, 1],

        "order": model.order,

        "train_rmse": train_rmse,
        "train_winkler": train_winkler
    }

    # ==========================================
    # Test Metrics
    # ==========================================

    if actual_future is not None:

        # -----------------------------
        # Test RMSE
        # -----------------------------

        test_rmse = np.sqrt(
            np.mean(
                (actual_future - forecast) ** 2
            )
        )

        # -----------------------------
        # Coverage
        # -----------------------------

        coverage = np.mean(
            (actual_future >= conf[:, 0]) &
            (actual_future <= conf[:, 1])
        )

        # -----------------------------
        # Test Winkler
        # -----------------------------

        test_winkler_scores = []

        for y, l, u in zip(
            actual_future,
            conf[:, 0],
            conf[:, 1]
        ):

            width = u - l

            if y < l:

                score = (
                    width +
                    (2 / alpha) * (l - y)
                )

            elif y > u:

                score = (
                    width +
                    (2 / alpha) * (y - u)
                )

            else:

                score = width

            test_winkler_scores.append(score)

        test_winkler = np.mean(
            test_winkler_scores
        )

        result["actual"] = actual_future

        result["test_rmse"] = test_rmse
        result["coverage"] = coverage
        result["test_winkler"] = test_winkler

    return result

In [ ]:
def evaluate_model(
    series,
    history=100,
    horizon=28,
    seasonal=True,
    m=7
):

    X, Y = make_sliding_windows(
        series,
        history,
        horizon
    )

    split = int(
        len(X) * 0.8
    )

    X_train = X[:split]
    Y_train = Y[:split]

    X_test = X[split:]
    Y_test = Y[split:]

    all_results = []

    orders = []

    naive_rmse_list = []

    for hist, future in zip(
        X_test,
        Y_test
    ):

        result = fit_autoarima(
            hist,
            horizon,
            future,
            seasonal,
            m
        )

        all_results.append(result)

        orders.append(
            result["order"]
        )

        naive_pred = naive_forecast(
            hist,
            horizon
        )

        naive_rmse = np.sqrt(
            np.mean(
                (future - naive_pred) ** 2
            )
        )

        naive_rmse_list.append(
            naive_rmse
        )

    return {
        "results": all_results,
        "orders": orders,
        "naive_rmse": np.mean(
            naive_rmse_list
        )
    }

In [ ]:
evaluation = evaluate_model(
    series,
    history=100,
    horizon=28,
    seasonal=True,
    m=7
)

results = evaluation["results"]

In [ ]:
print("Avg Train RMSE")

print(
    np.mean(
        [r["train_rmse"] for r in results]
    )
)

print("Avg Test RMSE")

print(
    np.mean(
        [r["test_rmse"] for r in results]
    )
)

print("Avg Coverage")

print(
    np.mean(
        [r["coverage"] for r in results]
    )
)

print("Avg Train Winkler")

print(
    np.mean(
        [r["train_winkler"] for r in results]
    )
)

print("Avg Test Winkler")

print(
    np.mean(
        [r["test_winkler"] for r in results]
    )
)

print("Naive RMSE")

print(
    evaluation["naive_rmse"]
)


In [ ]:
pd.Series(
    evaluation["orders"]
).value_counts()

In [ ]:
# ==========================================
# CONFORMAL CALIBRATION
# ==========================================

history = 100
horizon = 28

y_log = np.log1p(series)

X_all, Y_all = make_sliding_windows(
    y_log,
    history,
    horizon
)

n = len(X_all)

train_end = int(n * 0.60)
cal_end = int(n * 0.80)

X_cal = X_all[train_end:cal_end]
Y_cal = Y_all[train_end:cal_end]

X_test = X_all[cal_end:]
Y_test = Y_all[cal_end:]

mean = auto_out["mean"]
std = auto_out["std"]

print("Calibration windows:", len(X_cal))
print("Test windows:", len(X_test))

In [ ]:
# ==========================================
# CALIBRATION FORECASTS
# ==========================================

lower_cal = []
upper_cal = []

for hist in X_cal:

    hist_scaled = (
        hist - mean
    ) / std

    best_aic = np.inf
    best_model = None

    for p in [0,1,2]:
        for d in [0,1]:
            for q in [0,1,2]:

                try:

                    model = ARIMA(
                        hist_scaled,
                        order=(p,d,q)
                    )

                    fit = model.fit()

                    if fit.aic < best_aic:

                        best_aic = fit.aic
                        best_model = fit

                except:
                    pass

    forecast = best_model.get_forecast(
        steps=horizon
    )

    conf = forecast.conf_int(
        alpha=0.10
    )

    lower = conf[:,0]
    upper = conf[:,1]

    lower = lower * std + mean
    upper = upper * std + mean

    lower_cal.append(lower)
    upper_cal.append(upper)

lower_cal = np.array(lower_cal)
upper_cal = np.array(upper_cal)

In [ ]:
# ==========================================
# CONFORMAL SCORES
# ==========================================

cal_scores = np.maximum(
    lower_cal - Y_cal,
    Y_cal - upper_cal
)

print(cal_scores.shape)

In [ ]:
# ==========================================
# QHAT PER HORIZON
# ==========================================

alpha = 0.10

qhat_per_horizon = []

for h in range(horizon):

    scores_h = cal_scores[:,h]

    qhat_h = np.quantile(
        scores_h,
        1-alpha,
        method="higher"
    )

    qhat_per_horizon.append(
        qhat_h
    )

qhat_per_horizon = np.array(
    qhat_per_horizon
)

print("qhat:")
print(qhat_per_horizon)

In [ ]:
# ==========================================
# CONFORMAL METRICS
# ==========================================

yt = []
lo_conf = []
up_conf = []

for r in auto_out["results_test"]:

    yt.extend(r["y_true"])

    lo_conf.extend(
        r["lower_conf"]
    )

    up_conf.extend(
        r["upper_conf"]
    )

yt = np.array(yt)
lo_conf = np.array(lo_conf)
up_conf = np.array(up_conf)

coverage_conf = np.mean(
    (yt >= lo_conf)
    &
    (yt <= up_conf)
)

winkler_conf = compute_winkler_arrays(
    yt,
    lo_conf,
    up_conf
)

print("\n===== CONFORMAL AUTOARIMA =====")

print(
    "Raw Coverage:",
    auto_out["coverage"]
)

print(
    "Conf Coverage:",
    coverage_conf
)

print(
    "Raw Winkler:",
    auto_out["winkler"]
)

print(
    "Conf Winkler:",
    winkler_conf
)

In [ ]:
conformal_metrics = []

for i, r in enumerate(
    auto_out["results_test"]
):

    lower = r["lower_conf"]
    upper = r["upper_conf"]

    pred = r["y_pred"]

    actual = r["y_true"]

    rmse = np.sqrt(
        np.mean(
            (actual - pred)**2
        )
    )

    coverage = np.mean(
        (actual >= lower)
        &
        (actual <= upper)
    )

    winkler = compute_winkler_arrays(
        actual,
        lower,
        upper
    )

    conformal_metrics.append({

        "idx": i,

        "rmse": rmse,

        "coverage": coverage,

        "winkler": winkler
    })

best_conf_idx = min(
    conformal_metrics,
    key=lambda x: x["rmse"]
)["idx"]

worst_conf_idx = max(
    conformal_metrics,
    key=lambda x: x["rmse"]
)["idx"]

print(
    "Best conformal window:",
    best_conf_idx
)

print(
    "Worst conformal window:",
    worst_conf_idx
)

In [ ]:
def plot_autoarima_conformal_window(
    auto_out,
    idx,
    title
):

    r = auto_out["results_test"][idx]

    history = r["history"]

    actual = r["y_true"]

    pred = r["y_pred"]

    lower = r["lower_conf"]

    upper = r["upper_conf"]

    history_dates = r["history_dates"]

    future_dates = r["future_dates"]

    rmse = np.sqrt(
        np.mean(
            (actual - pred) ** 2
        )
    )

    coverage = np.mean(
        (actual >= lower)
        &
        (actual <= upper)
    )

    winkler = compute_winkler_arrays(
        actual,
        lower,
        upper
    )

    plt.figure(figsize=(14,6))

    plt.plot(
        history_dates,
        history,
        color="black",
        linewidth=2,
        label="History"
    )

    plt.plot(
        future_dates,
        actual,
        color="blue",
        linewidth=2,
        label="Actual"
    )

    plt.plot(
        future_dates,
        pred,
        color="red",
        linewidth=2,
        label="Forecast"
    )

    plt.fill_between(
        future_dates,
        lower,
        upper,
        alpha=0.25,
        color="green",
        label="Conformal PI"
    )

    plt.axvline(
        history_dates[-1],
        color="red",
        linestyle="--"
    )

    plt.title(
        f"{title}\n"
        f"RMSE={rmse:.3f} | "
        f"Coverage={coverage:.3f} | "
        f"Winkler={winkler:.3f}"
    )

    plt.xticks(rotation=45)

    plt.legend()

    plt.tight_layout()

    plt.show()

In [ ]:
plot_autoarima_conformal_window(
    auto_out=auto_out,
    idx=best_conf_idx,
    title="AutoARIMA Best Conformal Forecast Window"
)

In [ ]:
plot_autoarima_conformal_window(
    auto_out=auto_out,
    idx=worst_conf_idx,
    title="AutoARIMA Worst Conformal Forecast Window"
)